In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import copy
from IPython.display import Image
from qutip import *
import time
import math

In [ ]:
dim = 2      # number of levels for each qubit
GHz = 1e8    #1e9 (0.1GHz)
ns = 1e-9
b = destroy(dim)                     # sigma minus operator

s0m = tensor(b, qeye(dim))           # sigma minus on 0
s0p = tensor(b.dag(), qeye(dim))     # sigma plus on 0
s1m = tensor(qeye(dim), b)           # sigma minus on 1
s1p = tensor(qeye(dim), b.dag())     # sigma plus on 1

omega = [2*np.pi*4.2*GHz, 2*np.pi*4.7*GHz]            # qubit transition frequencies [rad/s]
Delta = [2*np.pi*0*GHz,2*np.pi*0*GHz] #[-.33, -.33]   # anharmonicities [rad/s]
Omega = [2*np.pi*1*GHz, 2*np.pi*1.1*GHz]              # control-qubit drive strength [rad/s]
J = 2*np.pi*0*GHz                                     # passive interaction [rad/s]

c_max = 10  # upper bound for control amplitudes
c_min = 0   # lower bound for control amplitudes
T = 10*ns   # target time
L = 2     # for Gaussian, always set to 2

H0 = ( omega[0]*s0p*s0m + omega[1]*s1p*s1m ) + \
     (1/2)*( Delta[0]*s0p*s0p*s0m*s0m + Delta[1]*s1p*s1p*s1m*s1m ) + \
     J*(s0p*s1m+s0m*s1p)

Hc0m = Omega[0]*(s0m)
Hc1m = Omega[1]*(s1m)
Hc0p = Omega[0]*(s0p)
Hc1p = Omega[1]*(s1p)

'''
# Piecewise Constant Control on qubit 0
def C0(t, args):
    c = args['c']     # list of control parameters [c11,...,c1L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[0][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[0][L-1]*float(t==t_inv[L])  )

def C1(t, args):
    c = args['c']     # list of control parameters [c21,...,c2L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[1][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[1][L-1]*float(t==t_inv[L])  )

# Piecewise Constant Control on qubit 1
def C2(t, args):
    c = args['c']     # list of control parameters [c31,...,c3L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[2][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[2][L-1]*float(t==t_inv[L])  )

def C3(t, args):
    c = args['c']     # list of control parameters [c41,...,c4L]
    T = args['T']     # target time 
    L = args['L']     # L pieces
    t_inv = list(np.linspace(0,T,L+1))     # endpoints of time interval
    return( np.sum([c[3][i] * ( float(t>=t_inv[i] and t<t_inv[i+1]) ) for i in range(L)])+c[3][L-1]*float(t==t_inv[L])  )
'''

# Gaussian Control on qubit 0
def C0(t, args):
    c = args['c']     # list of control parameters [c11, c12]
    T = args['T']     # target time
    return( c[0][0] * math.exp(-(t-(T/2))**2/((c[0][1])**2) * ( float(t>=0 and t<=T) ) ))
           
def C1(t, args):
    c = args['c']     # list of control parameters [c21, c22]
    T = args['T']     # target time
    return( c[1][0] * math.exp(-(t-(T/2))**2/((c[1][1])**2) * ( float(t>=0 and t<=T) ) ))

# Gaussian Control on qubit 1
def C2(t, args):
    c = args['c']     # list of control parameters [c11, c12]
    T = args['T']     # target time
    return( c[2][0] * math.exp(-(t-(T/2))**2/((c[2][1])**2) * ( float(t>=0 and t<=T) ) ))
           
def C3(t, args):
    c = args['c']     # list of control parameters [c21, c22]
    T = args['T']     # target time
    return( c[3][0] * math.exp(-(t-(T/2))**2/((c[3][1])**2) * ( float(t>=0 and t<=T) ) ))     
           

# Marker
def ep0(t, args):
    omega = args['omega']
    return np.exp(1j *omega[0]*t)

def em0(t, args):
    omega = args['omega']
    return np.exp(-1j *omega[0]*t)

def ep1(t, args):
    omega = args['omega']
    return np.exp(1j *omega[1]*t)

def em1(t, args):
    omega = args['omega']
    return np.exp(-1j *omega[1]*t)

# Control Functions C_i_j_k: i=0,1; j=m,p related to sm/sp; k=m,p related to (sm-sp) or (sm+sp)
def C0mm(t, args):
    return 1j*C0(t, args)*ep0(t, args)
def C0pm(t, args):
    return -1j*C0(t, args)*em0(t, args)

def C0mp(t, args):
    return C1(t, args)*ep0(t, args)
def C0pp(t, args):
    return C1(t, args)*em0(t, args)

def C1mm(t, args):
    return 1j*C2(t, args)*ep1(t, args)
def C1pm(t, args):
    return -1j*C2(t, args)*em1(t, args)

def C1mp(t, args):
    return C3(t, args)*ep1(t, args)
def C1pp(t, args):
    return C3(t, args)*em1(t, args)

# Total Hamiltonain
H = [H0, [Hc0m,C0mm], [Hc0p,C0pm], [Hc0m,C0mp], [Hc0p,C0pp], [Hc1m,C1mm], [Hc1p,C1pm], [Hc1m,C1mp], [Hc1p,C1pp]]   

In [ ]:
GA = sigmax() # Target Unitary on qubit 0
GB = sigmax() # Target Unitary on qubit 1

In [ ]:
def OBJ(c,T,L,omega,H,GA,GB):
    
    args = {
        'c': c,
        'T': T,
        'L': L,
        'omega': omega
    }
    
    U = propagator(H, T, args=args,options=Options(nsteps=1700))
    
    # SVD for partial traces
    sigma = np.sqrt((((tensor(GA,qeye(dim)).dag()*U).ptrace(1))*((tensor(GA,qeye(dim)).dag()*U).ptrace(1)).dag()).eigenstates()[0])
    lambd = np.sqrt((((tensor(qeye(dim),GB).dag()*U).ptrace(0))*((tensor(qeye(dim),GB).dag()*U).ptrace(0)).dag()).eigenstates()[0])
    
    # sum of fidelity
    return sum(sigma)/(2*dim**2)+sum(lambd)/(2*dim**2)

In [ ]:
# Debug/fix: scaled finite-difference Gaussian GRAPE optimizer
# This is not time-bin GRAPE. It optimizes the 8 Gaussian pulse parameters directly:
# x_scaled = [A0, width0_ns, A1, width1_ns, A2, width2_ns, A3, width3_ns].
# Width scaling into ns is essential because physical widths are around 1e-9 seconds.
# Signed amplitudes are allowed here because quadrature controls may need either sign.
# If eta=0 still cannot reach high fidelity after these fixes, likely explanations include:
# one Gaussian per channel is too restrictive, T is too short, amplitude bounds are too tight,
# the SUM objective differs from standard trace gate fidelity, or richer pulse families are needed.

# Gaussian-specific bounds for this debug optimizer. These are separate from old c_min/c_max.
gaussian_amp_min = -10.0
gaussian_amp_max = 10.0

gaussian_width_factor_min = 0.05
gaussian_width_factor_max = 1.50
gaussian_width_min = gaussian_width_factor_min * T
gaussian_width_max = gaussian_width_factor_max * T
gaussian_width_min_ns = gaussian_width_min / ns
gaussian_width_max_ns = gaussian_width_max / ns


def set_gaussian_width_bounds_from_T(T_use, factor_min=0.05, factor_max=1.50):
    """Return Gaussian width bounds for a chosen T without mutating global T."""
    T_use = float(T_use)
    width_min = float(factor_min) * T_use
    width_max = float(factor_max) * T_use
    return {
        'T': T_use,
        'factor_min': float(factor_min),
        'factor_max': float(factor_max),
        'width_min': width_min,
        'width_max': width_max,
        'width_min_ns': width_min / ns,
        'width_max_ns': width_max / ns,
    }


def _make_gaussian_scaled_bounds(T_use=None, amp_bounds=None, width_factor_bounds=None):
    """Build a single bounds dictionary for scaled Gaussian variables."""
    if T_use is None:
        T_use = T
    if amp_bounds is None:
        amp_bounds = (gaussian_amp_min, gaussian_amp_max)
    if width_factor_bounds is None:
        width_factor_bounds = (gaussian_width_factor_min, gaussian_width_factor_max)

    width_bounds = set_gaussian_width_bounds_from_T(
        T_use,
        factor_min=width_factor_bounds[0],
        factor_max=width_factor_bounds[1],
    )
    width_bounds.update({
        'amp_min': float(amp_bounds[0]),
        'amp_max': float(amp_bounds[1]),
        'amp_bounds': (float(amp_bounds[0]), float(amp_bounds[1])),
        'width_factor_bounds': (float(width_factor_bounds[0]), float(width_factor_bounds[1])),
    })
    return width_bounds


def gaussian_pack_scaled(c, ns_value=None):
    """
    Input:
        c = [[A0,width0_seconds], ..., [A3,width3_seconds]]
    Output:
        x_scaled = [A0,width0_ns,A1,width1_ns,...]
    """
    if ns_value is None:
        ns_value = ns
    c_array = np.asarray(c, dtype=float)
    if c_array.size != 8:
        raise ValueError('Gaussian c must contain 8 values and reshape to (4, 2).')
    c_array = c_array.reshape(4, 2)
    x_scaled = np.zeros(8, dtype=float)
    x_scaled[0::2] = c_array[:, 0]
    x_scaled[1::2] = c_array[:, 1] / ns_value
    return x_scaled


def gaussian_unpack_scaled(x_scaled, ns_value=None):
    """
    Input:
        x_scaled = [A0,width0_ns,A1,width1_ns,...]
    Output:
        c = [[A0,width0_seconds], ..., [A3,width3_seconds]]
    """
    if ns_value is None:
        ns_value = ns
    x_array = np.asarray(x_scaled, dtype=float)
    if x_array.size != 8:
        raise ValueError('Scaled Gaussian x must contain 8 values and reshape to (4, 2).')
    x_array = x_array.reshape(4, 2)
    c_array = np.zeros((4, 2), dtype=float)
    c_array[:, 0] = x_array[:, 0]
    c_array[:, 1] = x_array[:, 1] * ns_value
    return c_array.tolist()


def gaussian_clip_scaled(x_scaled, bounds_scaled):
    """
    Clip amplitudes to [gaussian_amp_min, gaussian_amp_max].
    Clip widths in ns to [width_min_ns, width_max_ns].
    """
    x_array = np.asarray(x_scaled, dtype=float).reshape(8).copy()
    x_array[0::2] = np.clip(x_array[0::2], bounds_scaled['amp_min'], bounds_scaled['amp_max'])
    x_array[1::2] = np.clip(x_array[1::2], bounds_scaled['width_min_ns'], bounds_scaled['width_max_ns'])
    return x_array


def _finite_real_float(value, invalid_value=-np.inf):
    try:
        value_float = float(np.real(value))
    except Exception:
        return invalid_value
    return value_float if np.isfinite(value_float) else invalid_value


def gaussian_fidelity_scaled(
    x_scaled,
    T_use=None,
    L_use=2,
    H_use=None,
    bounds_scaled=None,
):
    """Evaluate the existing OBJ on clipped scaled Gaussian coordinates."""
    if T_use is None:
        T_use = T
    if H_use is None:
        H_use = H
    if bounds_scaled is None:
        bounds_scaled = _make_gaussian_scaled_bounds(T_use=T_use)

    try:
        x_clipped = gaussian_clip_scaled(x_scaled, bounds_scaled)
        c_trial = gaussian_unpack_scaled(x_clipped)
        fidelity_value = OBJ(c_trial, T_use, L_use, omega, H_use, GA, GB)
    except Exception:
        return -np.inf
    return _finite_real_float(fidelity_value, invalid_value=-np.inf)


def _random_gaussian_x_scaled(rng, T_use, bounds_scaled):
    x_scaled = np.zeros(8, dtype=float)
    x_scaled[0::2] = rng.uniform(bounds_scaled['amp_min'], bounds_scaled['amp_max'], 4)
    x_scaled[1::2] = rng.uniform(0.20 * T_use / ns, 1.00 * T_use / ns, 4)
    return gaussian_clip_scaled(x_scaled, bounds_scaled)


def _central_difference_gradient_scaled(x_scaled, fidelity_current, dx, T_use, H_use, bounds_scaled):
    grad = np.zeros_like(x_scaled, dtype=float)
    invalid_count = 0

    for idx in range(len(x_scaled)):
        x_plus = np.asarray(x_scaled, dtype=float).copy()
        x_minus = np.asarray(x_scaled, dtype=float).copy()
        x_plus[idx] += 0.5 * dx
        x_minus[idx] -= 0.5 * dx
        x_plus = gaussian_clip_scaled(x_plus, bounds_scaled)
        x_minus = gaussian_clip_scaled(x_minus, bounds_scaled)

        denom = x_plus[idx] - x_minus[idx]
        if denom <= 0:
            continue

        f_plus = gaussian_fidelity_scaled(x_plus, T_use=T_use, L_use=2, H_use=H_use, bounds_scaled=bounds_scaled)
        f_minus = gaussian_fidelity_scaled(x_minus, T_use=T_use, L_use=2, H_use=H_use, bounds_scaled=bounds_scaled)

        if np.isfinite(f_plus) and np.isfinite(f_minus):
            grad[idx] = (f_plus - f_minus) / denom
        else:
            invalid_count += 1

    return grad, invalid_count


def Gaussian_GRAPE_scaled(
    T_use=None,
    H_use=None,
    x0_scaled=None,
    seed=None,
    max_iter=500,
    eps0=0.05,
    dx=1e-3,
    threshold=1e-7,
    success_threshold=0.99,
    amp_bounds=(-10.0, 10.0),
    width_factor_bounds=(0.05, 1.50),
    verbose=True,
):
    """Finite-difference gradient ascent over the 8 scaled Gaussian parameters."""
    if T_use is None:
        T_use = T
    T_use = float(T_use)
    if H_use is None:
        H_use = H

    bounds_scaled = _make_gaussian_scaled_bounds(
        T_use=T_use,
        amp_bounds=amp_bounds,
        width_factor_bounds=width_factor_bounds,
    )
    rng = np.random.default_rng(seed)

    if x0_scaled is None:
        x_scaled = _random_gaussian_x_scaled(rng, T_use, bounds_scaled)
    else:
        x_scaled = gaussian_clip_scaled(np.asarray(x0_scaled, dtype=float), bounds_scaled)
    x0_scaled_initial = np.asarray(x_scaled, dtype=float).copy()

    eps = float(eps0)
    dx = float(dx)
    record_cost = []
    stop_reason = 'max_iter_reached'
    count = 0
    invalid_gradient_evals = 0
    min_step = 1e-10

    current_fidelity = gaussian_fidelity_scaled(
        x_scaled,
        T_use=T_use,
        L_use=2,
        H_use=H_use,
        bounds_scaled=bounds_scaled,
    )
    initial_fidelity = current_fidelity
    record_cost.append(current_fidelity)

    if verbose:
        print(f'initial fidelity={current_fidelity:.8f}')

    if not np.isfinite(current_fidelity):
        stop_reason = 'invalid_initial_fidelity'
    elif current_fidelity >= success_threshold:
        stop_reason = 'success_threshold_reached'
    else:
        for iteration in range(int(max_iter)):
            count = iteration + 1

            grad, invalid_count = _central_difference_gradient_scaled(
                x_scaled,
                current_fidelity,
                dx,
                T_use,
                H_use,
                bounds_scaled,
            )
            invalid_gradient_evals += invalid_count
            grad_norm = np.linalg.norm(grad)

            if (not np.isfinite(grad_norm)) or grad_norm == 0:
                stop_reason = 'invalid_gradient'
                break

            direction = grad / grad_norm
            accepted = False

            for _ in range(20):
                x_trial = gaussian_clip_scaled(x_scaled + eps * direction, bounds_scaled)
                trial_fidelity = gaussian_fidelity_scaled(
                    x_trial,
                    T_use=T_use,
                    L_use=2,
                    H_use=H_use,
                    bounds_scaled=bounds_scaled,
                )

                if np.isfinite(trial_fidelity) and trial_fidelity > current_fidelity:
                    improvement = trial_fidelity - current_fidelity
                    x_scaled = x_trial
                    current_fidelity = trial_fidelity
                    record_cost.append(current_fidelity)
                    accepted = True

                    if verbose:
                        print(
                            f'iter {count}: fidelity={current_fidelity:.8f}, '
                            f'improvement={improvement:.3e}, eps={eps:.3e}, dx={dx:.3e}'
                        )

                    if current_fidelity >= success_threshold:
                        stop_reason = 'success_threshold_reached'
                    elif improvement < threshold:
                        stop_reason = 'improvement_below_threshold'
                    else:
                        stop_reason = 'max_iter_reached'
                    break

                eps *= 0.5
                dx *= 0.5
                if eps < min_step or dx < min_step:
                    stop_reason = 'step_size_too_small'
                    break

            if not accepted:
                if stop_reason == 'max_iter_reached':
                    stop_reason = 'line_search_failed'
                break

            if stop_reason in ('success_threshold_reached', 'improvement_below_threshold', 'step_size_too_small'):
                break

    x_opt_scaled = gaussian_clip_scaled(x_scaled, bounds_scaled)
    c_opt = gaussian_unpack_scaled(x_opt_scaled)
    final_fidelity = gaussian_fidelity_scaled(
        x_opt_scaled,
        T_use=T_use,
        L_use=2,
        H_use=H_use,
        bounds_scaled=bounds_scaled,
    )

    return {
        'record_cost': record_cost,
        'initial_fidelity': initial_fidelity,
        'x0_scaled': x0_scaled_initial,
        'x_opt_scaled': x_opt_scaled,
        'c_opt': c_opt,
        'final_fidelity': final_fidelity,
        'count': int(count),
        'success': bool(np.isfinite(final_fidelity) and final_fidelity >= success_threshold),
        'stop_reason': stop_reason,
        'T': float(T_use),
        'width_factor_bounds': tuple(width_factor_bounds),
        'amp_bounds': tuple(amp_bounds),
        'bounds_scaled': bounds_scaled,
        'eps_final': float(eps),
        'dx_final': float(dx),
        'invalid_gradient_evals': int(invalid_gradient_evals),
    }


def physical_eta0_initial_guesses_scaled(T_use=None, width_factor=0.5):
    """Construct unique eta=0 X-tensor-X inspired seeds using mostly C1 and C3 channels."""
    if T_use is None:
        T_use = T
    bounds_scaled = _make_gaussian_scaled_bounds(T_use=T_use)
    w_ns = float(width_factor) * float(T_use) / ns

    # The four sign variants used earlier are symmetry-equivalent at eta=0 and produced
    # duplicate trajectories. Keep one representative per amplitude scale.
    seeds = []
    for amp in [0.5, 1.0, 2.5, 5.0, 7.5]:
        seeds.append([0.0, w_ns, +amp, w_ns, 0.0, w_ns, +amp, w_ns])

    # Small nonzero C0/C2 seeds produced two unique relative-sign outcomes for each
    # small amplitude magnitude. Keep those representatives and omit sign duplicates.
    for small in [0.25, 0.50]:
        seeds.append([+small, w_ns, +5.0, w_ns, +small, w_ns, +5.0, w_ns])
        seeds.append([+small, w_ns, -5.0, w_ns, +small, w_ns, -5.0, w_ns])

    return [gaussian_clip_scaled(seed, bounds_scaled) for seed in seeds]


def _gaussian_scaled_at_bounds(x_scaled, bounds_scaled, amp_tol=1e-6, width_tol_ns=1e-6):
    x_scaled = np.asarray(x_scaled, dtype=float)
    amp_values = x_scaled[0::2]
    width_values = x_scaled[1::2]
    amp_at_bound = np.any(np.isclose(amp_values, bounds_scaled['amp_min'], atol=amp_tol)) or np.any(
        np.isclose(amp_values, bounds_scaled['amp_max'], atol=amp_tol)
    )
    width_at_bound = np.any(np.isclose(width_values, bounds_scaled['width_min_ns'], atol=width_tol_ns)) or np.any(
        np.isclose(width_values, bounds_scaled['width_max_ns'], atol=width_tol_ns)
    )
    return bool(amp_at_bound or width_at_bound)


def _summary_stats_from_records(records, success_threshold):
    final_fidelities = []
    for record in records:
        fidelity = _finite_real_float(record.get('final_fidelity'), invalid_value=-np.inf)
        if np.isfinite(fidelity):
            final_fidelities.append(fidelity)
    final_fidelities = np.asarray(final_fidelities, dtype=float)

    success_count = int(np.sum(final_fidelities >= success_threshold)) if final_fidelities.size else 0
    return {
        'best_fidelity': float(np.max(final_fidelities)) if final_fidelities.size else None,
        'mean_fidelity': float(np.mean(final_fidelities)) if final_fidelities.size else None,
        'median_fidelity': float(np.median(final_fidelities)) if final_fidelities.size else None,
        'success_count': success_count,
        'success_rate': success_count / len(records) if records else 0.0,
    }


def run_eta0_gaussian_grape_debug(
    n_restarts=30,
    seed_base=2026,
    T_use=None,
    width_factor_bounds=(0.05, 1.50),
    success_threshold=0.99,
    include_physical_seeds=True,
    make_plots=True,
    verbose=True,
):
    """Run eta=0 restarts with the transparent scaled finite-difference optimizer."""
    if T_use is None:
        T_use = T
    T_use = float(T_use)

    if 'build_H_with_crosstalk' in globals() and callable(build_H_with_crosstalk):
        H_eta0 = build_H_with_crosstalk(0.0)
    else:
        H_eta0 = H

    physical_seeds = physical_eta0_initial_guesses_scaled(T_use=T_use) if include_physical_seeds else []
    records = []

    for restart_id in range(int(n_restarts)):
        seed = int(seed_base) + restart_id
        x0_scaled = physical_seeds[restart_id] if restart_id < len(physical_seeds) else None
        used_physical_seed = x0_scaled is not None

        result = Gaussian_GRAPE_scaled(
            T_use=T_use,
            H_use=H_eta0,
            x0_scaled=x0_scaled,
            seed=seed,
            max_iter=500,
            eps0=0.05,
            dx=1e-3,
            threshold=1e-7,
            success_threshold=success_threshold,
            amp_bounds=(gaussian_amp_min, gaussian_amp_max),
            width_factor_bounds=width_factor_bounds,
            verbose=False,
        )
        bounds_scaled = result['bounds_scaled']
        record = {
            'restart': int(restart_id),
            'seed': int(seed),
            'used_physical_seed': bool(used_physical_seed),
            'initial_fidelity': result['initial_fidelity'],
            'final_fidelity': result['final_fidelity'],
            'success': result['success'],
            'stop_reason': result['stop_reason'],
            'count': result['count'],
            'x_opt_scaled': result['x_opt_scaled'],
            'c_opt': result['c_opt'],
            'record_cost': result['record_cost'],
            'eps_final': result['eps_final'],
            'dx_final': result['dx_final'],
            'invalid_gradient_evals': result['invalid_gradient_evals'],
            'ended_at_bounds': _gaussian_scaled_at_bounds(result['x_opt_scaled'], bounds_scaled),
        }
        records.append(record)

        if verbose:
            print(
                f'restart {restart_id:03d}, initial {record["initial_fidelity"]:.6f}, '
                f'final {record["final_fidelity"]:.6f}, success {record["success"]}, '
                f'stop {record["stop_reason"]}, iterations {record["count"]}'
            )

    summary = _summary_stats_from_records(records, success_threshold=success_threshold)
    summary.update({
        'n_restarts': int(n_restarts),
        'success_threshold': float(success_threshold),
        'failed_gradient_runs': int(sum(record['stop_reason'] == 'invalid_gradient' for record in records)),
        'runs_ending_at_bounds': int(sum(record['ended_at_bounds'] for record in records)),
        'T': float(T_use),
        'T_ns': float(T_use / ns),
        'width_factor_bounds': tuple(width_factor_bounds),
    })

    best_text = f'{summary["best_fidelity"]:.6f}' if summary['best_fidelity'] is not None else 'None'
    mean_text = f'{summary["mean_fidelity"]:.6f}' if summary['mean_fidelity'] is not None else 'None'
    median_text = f'{summary["median_fidelity"]:.6f}' if summary['median_fidelity'] is not None else 'None'
    print(
        'eta=0 Gaussian_GRAPE_scaled summary: '
        f'best={best_text}, mean={mean_text}, '
        f'median={median_text}, success={summary["success_count"]}/{n_restarts}, '
        f'success_rate={summary["success_rate"]:.3f}, failed_gradient_runs={summary["failed_gradient_runs"]}, '
        f'runs_ending_at_bounds={summary["runs_ending_at_bounds"]}'
    )

    if make_plots:
        final_fidelities = [record['final_fidelity'] for record in records]
        plt.figure(figsize=(8, 4))
        plt.plot(final_fidelities, marker='o', linestyle='none')
        plt.axhline(success_threshold, color='red', linestyle='--', label=f'threshold {success_threshold}')
        plt.xlabel('Restart')
        plt.ylabel('Final fidelity')
        plt.title('eta=0 scaled Gaussian finite-difference restarts')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

        plt.figure(figsize=(8, 4))
        for record in records:
            plt.plot(record['record_cost'], alpha=0.45)
        plt.axhline(success_threshold, color='red', linestyle='--', label=f'threshold {success_threshold}')
        plt.xlabel('Accepted step')
        plt.ylabel('Fidelity')
        plt.title('eta=0 convergence traces')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

    return {
        'records': records,
        'summary': summary,
    }


In [ ]:
# Eta=0 diagnostic scans and sanity checks

def scan_eta0_T_and_width_ranges(
    T_values,
    width_factor_ranges,
    n_restarts_per_setting=10,
    seed_base=2026,
    success_threshold=0.99,
):
    """Scan eta=0 performance over candidate gate times and width-factor ranges."""
    import pandas as pd

    rows = []
    for T_use in T_values:
        for width_factor_bounds in width_factor_ranges:
            print(
                f'\nScanning T={float(T_use / ns):.3f} ns, '
                f'width factors={width_factor_bounds}, restarts={n_restarts_per_setting}'
            )
            debug_result = run_eta0_gaussian_grape_debug(
                n_restarts=n_restarts_per_setting,
                seed_base=seed_base,
                T_use=T_use,
                width_factor_bounds=width_factor_bounds,
                success_threshold=success_threshold,
                include_physical_seeds=True,
                make_plots=False,
                verbose=False,
            )
            summary = debug_result['summary']
            rows.append({
                'T': float(T_use),
                'T_ns': float(T_use / ns),
                'width_factor_min': float(width_factor_bounds[0]),
                'width_factor_max': float(width_factor_bounds[1]),
                'width_range_label': f'{width_factor_bounds[0]:.2f}-{width_factor_bounds[1]:.2f}',
                'n_restarts': int(n_restarts_per_setting),
                'best_fidelity': summary['best_fidelity'],
                'mean_fidelity': summary['mean_fidelity'],
                'median_fidelity': summary['median_fidelity'],
                'success_count': summary['success_count'],
                'success_rate': summary['success_rate'],
                'failed_gradient_runs': summary['failed_gradient_runs'],
                'runs_ending_at_bounds': summary['runs_ending_at_bounds'],
            })

    df = pd.DataFrame(rows)

    if len(df):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
        for label, group in df.groupby('width_range_label'):
            group = group.sort_values('T_ns')
            axes[0].plot(group['T_ns'], group['best_fidelity'], marker='o', label=label)
            axes[1].plot(group['T_ns'], group['success_rate'], marker='o', label=label)

        axes[0].axhline(success_threshold, color='red', linestyle='--', alpha=0.7)
        axes[0].set_xlabel('T (ns)')
        axes[0].set_ylabel('Best fidelity')
        axes[0].set_title('Best eta=0 fidelity')
        axes[0].grid(True, alpha=0.3)

        axes[1].set_xlabel('T (ns)')
        axes[1].set_ylabel('Success rate')
        axes[1].set_title('eta=0 success rate')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(title='width factors')
        plt.tight_layout()
        plt.show()

    return df


def sanity_check_sum_objective_on_target():
    """Evaluate the same SUM-objective partial-trace logic on the exact target tensor(GA, GB)."""
    U_target = tensor(GA, GB)
    sigma = np.sqrt((
        ((tensor(GA, qeye(dim)).dag() * U_target).ptrace(1))
        * ((tensor(GA, qeye(dim)).dag() * U_target).ptrace(1)).dag()
    ).eigenstates()[0])
    lambd = np.sqrt((
        ((tensor(qeye(dim), GB).dag() * U_target).ptrace(0))
        * ((tensor(qeye(dim), GB).dag() * U_target).ptrace(0)).dag()
    ).eigenstates()[0])
    objective_value = float(np.real(sum(sigma) / (2 * dim**2) + sum(lambd) / (2 * dim**2)))
    print(f'SUM objective on exact tensor(GA, GB): {objective_value:.12f}')
    print('Expected value: approximately 1.0')
    return objective_value


def sanity_check_gaussian_control_shapes(c_test=None, T_use=None):
    """Plot C0-C3 over [0,T_use] to verify amplitudes and widths are used as Gaussian parameters."""
    if T_use is None:
        T_use = T
    T_use = float(T_use)

    if c_test is None:
        seed = physical_eta0_initial_guesses_scaled(T_use=T_use, width_factor=0.5)[0]
        c_test = gaussian_unpack_scaled(seed)
    else:
        c_array = np.asarray(c_test, dtype=float)
        if c_array.size != 8:
            raise ValueError('c_test must contain 8 values and reshape to (4, 2).')
        c_test = c_array.reshape(4, 2).tolist()

    args = {
        'c': c_test,
        'T': T_use,
        'L': 2,
        'omega': omega,
    }
    t_values = np.linspace(0, T_use, 300)
    controls = [
        [C0(t, args) for t in t_values],
        [C1(t, args) for t in t_values],
        [C2(t, args) for t in t_values],
        [C3(t, args) for t in t_values],
    ]

    plt.figure(figsize=(8, 4))
    for idx, values in enumerate(controls):
        plt.plot(t_values / ns, values, label=f'C{idx}')
    plt.xlabel('t (ns)')
    plt.ylabel('Control envelope')
    plt.title('Gaussian control envelope sanity check')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print('c_test [amplitude, width(ns)]:')
    print(np.array([[row[0], row[1] / ns] for row in c_test]))
    return c_test


In [ ]:
# Clean Gaussian crosstalk Hamiltonian and JSON helpers
# This cell keeps the same control Hamiltonian structure as H and only swaps H[0] for H0(eta).

import json
import pickle
from pathlib import Path


def make_json_safe(obj):
    """Recursively convert numpy and pathlib objects into JSON-safe Python values."""
    if isinstance(obj, dict):
        return {str(make_json_safe(key)): make_json_safe(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(value) for value in obj]
    if isinstance(obj, np.ndarray):
        return make_json_safe(obj.tolist())
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return value if np.isfinite(value) else None
    if isinstance(obj, (np.complexfloating, complex)):
        return {
            'real': make_json_safe(float(np.real(obj))),
            'imag': make_json_safe(float(np.imag(obj))),
        }
    if isinstance(obj, Path):
        return str(obj)
    return obj


def build_H0_with_crosstalk(eta):
    """Build H0(eta) with exchange/coupling eta using the existing notebook operators."""
    eta = float(eta)
    H0_eta = omega[0] * s0p * s0m + omega[1] * s1p * s1m

    delta_values = globals().get('Delta', [0.0, 0.0])
    try:
        delta_array = np.asarray(delta_values, dtype=float).reshape(-1)
        delta0 = float(delta_array[0]) if delta_array.size > 0 else 0.0
        delta1 = float(delta_array[1]) if delta_array.size > 1 else 0.0
    except Exception:
        delta0 = 0.0
        delta1 = 0.0

    H0_eta = H0_eta + 0.5 * (
        delta0 * s0p * s0p * s0m * s0m
        + delta1 * s1p * s1p * s1m * s1m
    )
    H0_eta = H0_eta + eta * (s0p * s1m + s0m * s1p)
    return H0_eta


def build_H_with_crosstalk(eta):
    """Return the current Hamiltonian structure with H[0] replaced by H0(eta)."""
    return [build_H0_with_crosstalk(eta)] + list(H[1:])


def eta_directory_name(eta_index):
    return f'eta_{int(eta_index):03d}'


def save_gaussian_grape_records(records, eta_dir):
    eta_dir = Path(eta_dir)
    eta_dir.mkdir(parents=True, exist_ok=True)
    json_path = eta_dir / 'results.json'
    pickle_path = eta_dir / 'results.pkl'

    with json_path.open('w', encoding='utf-8') as json_file:
        json.dump(make_json_safe(records), json_file, indent=2, allow_nan=False)

    with pickle_path.open('wb') as pickle_file:
        pickle.dump(records, pickle_file)

    return json_path, pickle_path


In [ ]:
# GRAPE-style Gaussian crosstalk experiment runner
# This production runner uses Gaussian_GRAPE_scaled, not L-BFGS-B.
# It optimizes the 8 Gaussian landscape coordinates [A0,w0_ns,A1,w1_ns,A2,w2_ns,A3,w3_ns].

# Final crosstalk grid used by the full Gaussian GRAPE sweep.
eta_values_crosstalk_grid = 2*np.pi*GHz*np.array([
    0.000,
    0.002,
    0.005,
    0.010,
    0.020,
    0.035,
    0.050,
    0.075,
    0.100,
    0.125,
    0.150,
    0.200,
    0.300,
    0.500,
    0.750,
    1.000,
])

# The eta=0 debug run showed T=10 ns and these width factors work well.
# T=50 ns produced ODE excess-work warnings with the current OBJ nsteps, so it is not a default production choice.
gaussian_grape_primary_T = 10 * ns
gaussian_grape_primary_width_factor_bounds = (0.05, 1.50)
gaussian_grape_amp_bounds = (-10.0, 10.0)
gaussian_grape_success_threshold = 0.999

# Final-sweep defaults. Increase n_restarts in the guarded run cell, not inside the optimizer.
gaussian_grape_max_iter = 2000
gaussian_grape_eps0 = 0.05
gaussian_grape_dx = 1e-3
gaussian_grape_improvement_threshold = 1e-8
gaussian_grape_analysis_thresholds = (0.99, 0.995, 0.999)
gaussian_grape_physical_seed_width_factor = 0.25

# Seed schedule used for the final full sweep: 20 physical seeds, 30 local/continuation seeds,
# and the remaining restarts as quasi-random Gaussian seeds.
gaussian_grape_n_physical_seeds = 20
gaussian_grape_n_bridge_seeds = 30
gaussian_grape_continuation_start_eta_index = 4


def _final_physical_seed_grid_scaled(T_use=None, width_factor=None, bounds_scaled=None):
    """Physical X tensor X seed grid with symmetry-related sign variants."""
    if T_use is None:
        T_use = gaussian_grape_primary_T
    if width_factor is None:
        width_factor = gaussian_grape_physical_seed_width_factor
    if bounds_scaled is None:
        bounds_scaled = _make_gaussian_scaled_bounds(
            T_use=T_use,
            amp_bounds=gaussian_grape_amp_bounds,
            width_factor_bounds=gaussian_grape_primary_width_factor_bounds,
        )

    w_ns = float(width_factor) * float(T_use) / ns
    seeds = []
    for amp in [0.5, 1.0, 2.5, 5.0, 7.5]:
        seeds.extend([
            [0.0, w_ns, +amp, w_ns, 0.0, w_ns, +amp, w_ns],
            [0.0, w_ns, -amp, w_ns, 0.0, w_ns, -amp, w_ns],
            [0.0, w_ns, +amp, w_ns, 0.0, w_ns, -amp, w_ns],
            [0.0, w_ns, -amp, w_ns, 0.0, w_ns, +amp, w_ns],
        ])
    return [gaussian_clip_scaled(seed, bounds_scaled) for seed in seeds]


def _perturb_gaussian_seed_scaled(base_seed, rng, bounds_scaled, amp_sigma=0.45, width_sigma_ns=0.45):
    """Make a bounded local perturbation in scaled Gaussian coordinates."""
    base_seed = np.asarray(base_seed, dtype=float)
    noise = np.zeros_like(base_seed)
    noise[0::2] = rng.normal(0.0, amp_sigma, size=4)
    noise[1::2] = rng.normal(0.0, width_sigma_ns, size=4)
    return gaussian_clip_scaled(base_seed + noise, bounds_scaled)


def _best_record_for_continuation(records):
    """Choose the previous eta record with the best finite final fidelity."""
    if not records:
        return None
    valid_records = [
        record for record in records
        if record.get('final_fidelity') is not None
        and np.isfinite(record.get('final_fidelity'))
        and record.get('x_opt_scaled') is not None
    ]
    if not valid_records:
        return None
    return max(valid_records, key=lambda record: record['final_fidelity'])


def _seed_spec(
    x0_scaled,
    seed_type,
    seed_subindex,
    parent_record=None,
):
    parent_record = parent_record or {}
    return {
        'x0_scaled': x0_scaled,
        'seed_type': seed_type,
        'seed_subindex': int(seed_subindex),
        'parent_eta_index': parent_record.get('eta_index'),
        'parent_restart': parent_record.get('restart'),
        'parent_seed': parent_record.get('seed'),
        'parent_final_fidelity': parent_record.get('final_fidelity'),
    }


def make_gaussian_grape_seed_schedule(
    eta_index,
    n_restarts,
    seed_base=2026,
    T_use=None,
    width_factor_bounds=None,
    amp_bounds=None,
    previous_eta_records=None,
    include_physical_seeds=True,
    use_continuation=True,
):
    """Build restart seed metadata for one eta.

    Physical seeds probe the eta=0-inspired X tensor X structure. For later eta values,
    bridge seeds continue from the previous eta's best optimum and local perturbations.
    Remaining restarts use the optimizer's reproducible quasi-random initializer.
    """
    if T_use is None:
        T_use = gaussian_grape_primary_T
    if width_factor_bounds is None:
        width_factor_bounds = gaussian_grape_primary_width_factor_bounds
    if amp_bounds is None:
        amp_bounds = gaussian_grape_amp_bounds

    bounds_scaled = _make_gaussian_scaled_bounds(
        T_use=T_use,
        amp_bounds=amp_bounds,
        width_factor_bounds=width_factor_bounds,
    )
    n_restarts = int(n_restarts)
    seed_base = int(seed_base)
    specs = []

    physical_seeds = _final_physical_seed_grid_scaled(
        T_use=T_use,
        bounds_scaled=bounds_scaled,
    ) if include_physical_seeds else []

    for seed_subindex, x0_scaled in enumerate(physical_seeds[:min(gaussian_grape_n_physical_seeds, n_restarts)]):
        specs.append(_seed_spec(x0_scaled, 'physical', seed_subindex))

    remaining = n_restarts - len(specs)
    bridge_count = min(gaussian_grape_n_bridge_seeds, remaining)
    parent_record = None
    bridge_base = physical_seeds[0] if physical_seeds else None
    bridge_type = 'local_physical_perturbation'

    if (
        use_continuation
        and eta_index >= gaussian_grape_continuation_start_eta_index
        and previous_eta_records
    ):
        parent_record = _best_record_for_continuation(previous_eta_records)
        if parent_record is not None:
            bridge_base = np.asarray(parent_record['x_opt_scaled'], dtype=float)
            bridge_type = 'continuation_perturbation'

    if bridge_base is not None:
        for bridge_subindex in range(bridge_count):
            restart_id = len(specs)
            rng = np.random.default_rng(seed_base + restart_id)
            if bridge_subindex == 0:
                x0_scaled = gaussian_clip_scaled(bridge_base, bounds_scaled)
                seed_type = 'continuation_exact' if bridge_type == 'continuation_perturbation' else bridge_type
            else:
                x0_scaled = _perturb_gaussian_seed_scaled(bridge_base, rng, bounds_scaled)
                seed_type = bridge_type
            specs.append(_seed_spec(x0_scaled, seed_type, bridge_subindex, parent_record=parent_record))

    while len(specs) < n_restarts:
        seed_subindex = len([spec for spec in specs if spec['seed_type'] == 'quasi_random'])
        specs.append(_seed_spec(None, 'quasi_random', seed_subindex))

    return specs


def _success_at_thresholds(final_fidelity, thresholds):
    if final_fidelity is None or not np.isfinite(final_fidelity):
        return {str(threshold): False for threshold in thresholds}
    return {str(threshold): bool(final_fidelity >= threshold) for threshold in thresholds}


def run_one_gaussian_grape_restart(
    eta,
    restart_id,
    seed,
    T_use=None,
    width_factor_bounds=None,
    amp_bounds=None,
    success_threshold=None,
    max_iter=None,
    eps0=None,
    dx=None,
    threshold=None,
    x0_scaled=None,
    seed_type='quasi_random',
    seed_subindex=0,
    parent_eta_index=None,
    parent_restart=None,
    parent_seed=None,
    parent_final_fidelity=None,
    analysis_thresholds=None,
):
    """Run one scaled finite-difference Gaussian GRAPE restart for one eta."""
    if T_use is None:
        T_use = gaussian_grape_primary_T
    if width_factor_bounds is None:
        width_factor_bounds = gaussian_grape_primary_width_factor_bounds
    if amp_bounds is None:
        amp_bounds = gaussian_grape_amp_bounds
    if success_threshold is None:
        success_threshold = gaussian_grape_success_threshold
    if max_iter is None:
        max_iter = gaussian_grape_max_iter
    if eps0 is None:
        eps0 = gaussian_grape_eps0
    if dx is None:
        dx = gaussian_grape_dx
    if threshold is None:
        threshold = gaussian_grape_improvement_threshold
    if analysis_thresholds is None:
        analysis_thresholds = gaussian_grape_analysis_thresholds

    H_eta = build_H_with_crosstalk(eta)
    result = Gaussian_GRAPE_scaled(
        T_use=T_use,
        H_use=H_eta,
        x0_scaled=x0_scaled,
        seed=seed,
        max_iter=max_iter,
        eps0=eps0,
        dx=dx,
        threshold=threshold,
        success_threshold=success_threshold,
        amp_bounds=amp_bounds,
        width_factor_bounds=width_factor_bounds,
        verbose=False,
    )

    final_fidelity = result['final_fidelity']
    record = {
        'eta': float(eta),
        'eta_factor': float(eta / (2*np.pi*GHz)),
        'restart': int(restart_id),
        'seed': int(seed),
        'seed_type': str(seed_type),
        'seed_subindex': int(seed_subindex),
        'used_physical_seed': x0_scaled is not None,
        'parent_eta_index': parent_eta_index,
        'parent_restart': parent_restart,
        'parent_seed': parent_seed,
        'parent_final_fidelity': parent_final_fidelity,
        'initial_fidelity': result['initial_fidelity'],
        'final_fidelity': final_fidelity,
        'success': result['success'],
        'success_threshold': float(success_threshold),
        'success_at_thresholds': _success_at_thresholds(final_fidelity, analysis_thresholds),
        'stop_reason': result['stop_reason'],
        'count': result['count'],
        'x0_scaled': result.get('x0_scaled'),
        'x_opt_scaled': result['x_opt_scaled'],
        'c_opt': result['c_opt'],
        'record_cost': result['record_cost'],
        'eps_final': result['eps_final'],
        'dx_final': result['dx_final'],
        'invalid_gradient_evals': result['invalid_gradient_evals'],
        'pulse_family': 'gaussian_amp_width_scaled_ns',
        'optimizer': 'finite_difference_gradient_ascent',
        'objective': 'current_OBJ',
        'target': 'X_tensor_X',
        'T': float(T_use),
        'T_ns': float(T_use / ns),
        'L': 2,
        'amp_bounds': tuple(amp_bounds),
        'width_factor_bounds': tuple(width_factor_bounds),
        'analysis_thresholds': tuple(analysis_thresholds),
        'optimizer_success_threshold': float(success_threshold),
        'max_iter': int(max_iter),
        'eps0': float(eps0),
        'dx0': float(dx),
        'improvement_threshold': float(threshold),
    }
    record['ended_at_bounds'] = _gaussian_scaled_at_bounds(result['x_opt_scaled'], result['bounds_scaled'])
    return record


def summarize_gaussian_grape_records(records, success_threshold=0.999):
    fidelities = np.asarray([
        record['final_fidelity']
        for record in records
        if record.get('final_fidelity') is not None and np.isfinite(record.get('final_fidelity'))
    ], dtype=float)
    success_count = int(np.sum(fidelities >= success_threshold)) if fidelities.size else 0
    stop_reason_counts = {}
    seed_type_counts = {}
    for record in records:
        stop_reason = record.get('stop_reason', 'unknown')
        stop_reason_counts[stop_reason] = stop_reason_counts.get(stop_reason, 0) + 1
        seed_type = record.get('seed_type', 'unknown')
        seed_type_counts[seed_type] = seed_type_counts.get(seed_type, 0) + 1

    return {
        'n_restarts': len(records),
        'best_fidelity': float(np.max(fidelities)) if fidelities.size else None,
        'mean_fidelity': float(np.mean(fidelities)) if fidelities.size else None,
        'median_fidelity': float(np.median(fidelities)) if fidelities.size else None,
        'success_count': success_count,
        'success_rate': success_count / len(records) if records else 0.0,
        'failed_gradient_runs': int(sum(record.get('stop_reason') == 'invalid_gradient' for record in records)),
        'runs_ending_at_bounds': int(sum(record.get('ended_at_bounds', False) for record in records)),
        'stop_reason_counts': stop_reason_counts,
        'seed_type_counts': seed_type_counts,
        'mean_iterations': float(np.mean([record.get('count', 0) for record in records])) if records else None,
        'median_iterations': float(np.median([record.get('count', 0) for record in records])) if records else None,
    }


def print_gaussian_grape_summary(label, summary):
    best_text = f'{summary["best_fidelity"]:.6f}' if summary['best_fidelity'] is not None else 'None'
    mean_text = f'{summary["mean_fidelity"]:.6f}' if summary['mean_fidelity'] is not None else 'None'
    median_text = f'{summary["median_fidelity"]:.6f}' if summary['median_fidelity'] is not None else 'None'
    mean_iter_text = f'{summary["mean_iterations"]:.1f}' if summary['mean_iterations'] is not None else 'None'
    print(
        f'{label}: best={best_text}, mean={mean_text}, median={median_text}, '
        f'success={summary["success_count"]}/{summary["n_restarts"]}, '
        f'success_rate={summary["success_rate"]:.3f}, '
        f'failed_gradient_runs={summary["failed_gradient_runs"]}, '
        f'runs_ending_at_bounds={summary["runs_ending_at_bounds"]}, '
        f'mean_iterations={mean_iter_text}'
    )
    print('stop reasons:', summary['stop_reason_counts'])
    print('seed types:', summary.get('seed_type_counts', {}))


def _failure_record_for_exception(
    eta,
    restart_id,
    seed,
    exc,
    seed_spec,
    T_use,
    width_factor_bounds,
    amp_bounds,
    success_threshold,
    max_iter,
    eps0,
    dx,
    threshold,
    analysis_thresholds,
):
    return {
        'eta': float(eta),
        'eta_factor': float(eta / (2*np.pi*GHz)),
        'restart': int(restart_id),
        'seed': int(seed),
        'seed_type': seed_spec.get('seed_type', 'unknown'),
        'seed_subindex': seed_spec.get('seed_subindex', 0),
        'used_physical_seed': seed_spec.get('x0_scaled') is not None,
        'parent_eta_index': seed_spec.get('parent_eta_index'),
        'parent_restart': seed_spec.get('parent_restart'),
        'parent_seed': seed_spec.get('parent_seed'),
        'parent_final_fidelity': seed_spec.get('parent_final_fidelity'),
        'initial_fidelity': None,
        'final_fidelity': None,
        'success': False,
        'success_threshold': float(success_threshold),
        'success_at_thresholds': _success_at_thresholds(None, analysis_thresholds),
        'stop_reason': f'exception: {exc}',
        'count': 0,
        'x0_scaled': seed_spec.get('x0_scaled'),
        'x_opt_scaled': None,
        'c_opt': None,
        'record_cost': [],
        'pulse_family': 'gaussian_amp_width_scaled_ns',
        'optimizer': 'finite_difference_gradient_ascent',
        'objective': 'current_OBJ',
        'target': 'X_tensor_X',
        'T': float(T_use),
        'T_ns': float(T_use / ns),
        'L': 2,
        'amp_bounds': tuple(amp_bounds),
        'width_factor_bounds': tuple(width_factor_bounds),
        'analysis_thresholds': tuple(analysis_thresholds),
        'optimizer_success_threshold': float(success_threshold),
        'max_iter': int(max_iter),
        'eps0': float(eps0),
        'dx0': float(dx),
        'improvement_threshold': float(threshold),
        'ended_at_bounds': False,
    }


def run_gaussian_grape_restarts_for_eta(
    eta,
    eta_index=0,
    n_restarts=50,
    seed_base=2026,
    save_dir='GaussianGRAPEResults_eta0_validation',
    T_use=None,
    width_factor_bounds=None,
    amp_bounds=None,
    success_threshold=None,
    max_iter=None,
    eps0=None,
    dx=None,
    threshold=None,
    include_physical_seeds=True,
    previous_eta_records=None,
    use_continuation=True,
    analysis_thresholds=None,
):
    """Run and save scaled Gaussian GRAPE restarts for a single eta."""
    if T_use is None:
        T_use = gaussian_grape_primary_T
    if width_factor_bounds is None:
        width_factor_bounds = gaussian_grape_primary_width_factor_bounds
    if amp_bounds is None:
        amp_bounds = gaussian_grape_amp_bounds
    if success_threshold is None:
        success_threshold = gaussian_grape_success_threshold
    if max_iter is None:
        max_iter = gaussian_grape_max_iter
    if eps0 is None:
        eps0 = gaussian_grape_eps0
    if dx is None:
        dx = gaussian_grape_dx
    if threshold is None:
        threshold = gaussian_grape_improvement_threshold
    if analysis_thresholds is None:
        analysis_thresholds = gaussian_grape_analysis_thresholds

    eta_dir = Path(save_dir) / eta_directory_name(eta_index)
    eta_dir.mkdir(parents=True, exist_ok=True)
    seed_specs = make_gaussian_grape_seed_schedule(
        eta_index=eta_index,
        n_restarts=n_restarts,
        seed_base=seed_base,
        T_use=T_use,
        width_factor_bounds=width_factor_bounds,
        amp_bounds=amp_bounds,
        previous_eta_records=previous_eta_records,
        include_physical_seeds=include_physical_seeds,
        use_continuation=use_continuation,
    )
    records = []

    for restart_id, seed_spec in enumerate(seed_specs):
        seed = int(seed_base) + int(restart_id)
        try:
            record = run_one_gaussian_grape_restart(
                eta=eta,
                restart_id=restart_id,
                seed=seed,
                T_use=T_use,
                width_factor_bounds=width_factor_bounds,
                amp_bounds=amp_bounds,
                success_threshold=success_threshold,
                max_iter=max_iter,
                eps0=eps0,
                dx=dx,
                threshold=threshold,
                x0_scaled=seed_spec.get('x0_scaled'),
                seed_type=seed_spec.get('seed_type', 'quasi_random'),
                seed_subindex=seed_spec.get('seed_subindex', 0),
                parent_eta_index=seed_spec.get('parent_eta_index'),
                parent_restart=seed_spec.get('parent_restart'),
                parent_seed=seed_spec.get('parent_seed'),
                parent_final_fidelity=seed_spec.get('parent_final_fidelity'),
                analysis_thresholds=analysis_thresholds,
            )
            record['eta_index'] = int(eta_index)
        except Exception as exc:
            record = _failure_record_for_exception(
                eta=eta,
                restart_id=restart_id,
                seed=seed,
                exc=exc,
                seed_spec=seed_spec,
                T_use=T_use,
                width_factor_bounds=width_factor_bounds,
                amp_bounds=amp_bounds,
                success_threshold=success_threshold,
                max_iter=max_iter,
                eps0=eps0,
                dx=dx,
                threshold=threshold,
                analysis_thresholds=analysis_thresholds,
            )
            record['eta_index'] = int(eta_index)

        records.append(record)
        save_gaussian_grape_records(records, eta_dir)

        fidelity = record.get('final_fidelity')
        fidelity_text = f'{fidelity:.6f}' if fidelity is not None and np.isfinite(fidelity) else 'None'
        initial = record.get('initial_fidelity')
        initial_text = f'{initial:.6f}' if initial is not None and np.isfinite(initial) else 'None'
        print(
            f'eta index {eta_index}, eta value {float(eta):.6e}, restart {restart_id:03d}, '
            f'seed_type {record.get("seed_type")}, initial {initial_text}, final {fidelity_text}, '
            f'success {record.get("success")}, stop {record.get("stop_reason")}, iterations {record.get("count")}'
        )

    summary = summarize_gaussian_grape_records(records, success_threshold=success_threshold)
    summary.update({
        'eta_index': int(eta_index),
        'eta': float(eta),
        'eta_factor': float(eta / (2*np.pi*GHz)),
        'success_threshold': float(success_threshold),
        'max_iter': int(max_iter),
        'n_restarts': int(n_restarts),
    })
    save_gaussian_grape_records(records, eta_dir)
    with (eta_dir / 'summary.json').open('w', encoding='utf-8') as summary_file:
        json.dump(make_json_safe(summary), summary_file, indent=2, allow_nan=False)
    print_gaussian_grape_summary(f'eta index {eta_index}, eta value {float(eta):.6e}', summary)
    return records, summary


def run_gaussian_grape_crosstalk_grid(
    eta_values,
    n_restarts=150,
    seed_base=2026,
    save_dir='GaussianGRAPEResults_crosstalk_final_v1',
    T_use=None,
    width_factor_bounds=None,
    amp_bounds=None,
    success_threshold=None,
    max_iter=None,
    eps0=None,
    dx=None,
    threshold=None,
    include_physical_seeds=True,
    use_continuation=True,
    analysis_thresholds=None,
):
    """Run the full crosstalk grid with the working scaled Gaussian GRAPE optimizer."""
    eta_values = list(eta_values)
    total_optimizations = len(eta_values) * int(n_restarts)
    print('=' * 72)
    print('Scaled Gaussian GRAPE crosstalk grid')
    print(f'eta values: {len(eta_values)}')
    print(f'restarts per eta: {n_restarts}')
    print(f'estimated total optimizations: {total_optimizations}')
    print(f'T: {float((T_use if T_use is not None else gaussian_grape_primary_T) / ns):.3f} ns')
    print(f'width factors: {width_factor_bounds if width_factor_bounds is not None else gaussian_grape_primary_width_factor_bounds}')
    print(f'success threshold: {success_threshold if success_threshold is not None else gaussian_grape_success_threshold}')
    print(f'max iterations: {max_iter if max_iter is not None else gaussian_grape_max_iter}')
    print(f'save directory: {save_dir}')
    print('=' * 72)

    all_results = {}
    summaries = []
    previous_eta_records = None
    for eta_index, eta in enumerate(eta_values):
        print(f'\nStarting eta index {eta_index}/{len(eta_values)-1}, eta value {float(eta):.6e}')
        records, summary = run_gaussian_grape_restarts_for_eta(
            eta=eta,
            eta_index=eta_index,
            n_restarts=n_restarts,
            seed_base=seed_base,
            save_dir=save_dir,
            T_use=T_use,
            width_factor_bounds=width_factor_bounds,
            amp_bounds=amp_bounds,
            success_threshold=success_threshold,
            max_iter=max_iter,
            eps0=eps0,
            dx=dx,
            threshold=threshold,
            include_physical_seeds=include_physical_seeds,
            previous_eta_records=previous_eta_records,
            use_continuation=use_continuation,
            analysis_thresholds=analysis_thresholds,
        )
        all_results[f'eta_{eta_index:03d}'] = records
        summaries.append(summary)
        previous_eta_records = records

        summary_path = Path(save_dir) / 'grid_summary.json'
        summary_path.parent.mkdir(parents=True, exist_ok=True)
        with summary_path.open('w', encoding='utf-8') as summary_file:
            json.dump(make_json_safe(summaries), summary_file, indent=2, allow_nan=False)

    print('\nScaled Gaussian GRAPE crosstalk grid complete.')
    return {
        'results': all_results,
        'summaries': summaries,
    }


In [ ]:
# Optional clean-notebook sanity checks
# Run these first if you want to verify the SUM objective and Gaussian envelope plumbing.

RUN_CLEAN_GAUSSIAN_SANITY_CHECKS = False

if RUN_CLEAN_GAUSSIAN_SANITY_CHECKS:
    sanity_check_sum_objective_on_target()
    sanity_check_gaussian_control_shapes()
else:
    print('Clean Gaussian sanity checks not started. Set RUN_CLEAN_GAUSSIAN_SANITY_CHECKS = True to run.')


In [ ]:
# Eta=0 validation run before the full crosstalk sweep
# Keep this guarded. Flip RUN_ETA0_GRAPE_VALIDATION to True when ready.

RUN_ETA0_GRAPE_VALIDATION = False

eta0_validation_n_restarts = 10
eta0_validation_seed_base = 2026
eta0_validation_save_dir = 'GaussianGRAPEResults_eta0_validation_v1'

if RUN_ETA0_GRAPE_VALIDATION:
    eta0_validation_records, eta0_validation_summary = run_gaussian_grape_restarts_for_eta(
        eta=0.0,
        eta_index=0,
        n_restarts=eta0_validation_n_restarts,
        seed_base=eta0_validation_seed_base,
        save_dir=eta0_validation_save_dir,
        T_use=gaussian_grape_primary_T,
        width_factor_bounds=gaussian_grape_primary_width_factor_bounds,
        amp_bounds=gaussian_grape_amp_bounds,
        success_threshold=gaussian_grape_success_threshold,
        max_iter=gaussian_grape_max_iter,
        eps0=gaussian_grape_eps0,
        dx=gaussian_grape_dx,
        threshold=gaussian_grape_improvement_threshold,
        include_physical_seeds=True,
    )
else:
    print('Eta=0 validation not started. Set RUN_ETA0_GRAPE_VALIDATION = True to run.')


In [ ]:
# Full crosstalk-grid run with scaled Gaussian GRAPE
# Run this only after eta=0 validation is healthy.

RUN_FULL_GRAPE_CROSSTALK_GRID = False

full_grid_n_restarts = 150  # Final-analysis default after successful eta=0 validation.
full_grid_seed_base = 2026
full_grid_save_dir = 'GaussianGRAPEResults_crosstalk_final_v1'

if RUN_FULL_GRAPE_CROSSTALK_GRID:
    gaussian_grape_crosstalk_grid_results = run_gaussian_grape_crosstalk_grid(
        eta_values=eta_values_crosstalk_grid,
        n_restarts=full_grid_n_restarts,
        seed_base=full_grid_seed_base,
        save_dir=full_grid_save_dir,
        T_use=gaussian_grape_primary_T,
        width_factor_bounds=gaussian_grape_primary_width_factor_bounds,
        amp_bounds=gaussian_grape_amp_bounds,
        success_threshold=gaussian_grape_success_threshold,
        max_iter=gaussian_grape_max_iter,
        eps0=gaussian_grape_eps0,
        dx=gaussian_grape_dx,
        threshold=gaussian_grape_improvement_threshold,
        include_physical_seeds=True,
    )
else:
    print('Full GRAPE crosstalk grid not started. Set RUN_FULL_GRAPE_CROSSTALK_GRID = True to run.')


In [ ]:
# Optional eta=0 T and width-range validation scan (Completed)

# RUN_ETA0_T_WIDTH_VALIDATION_SCAN = False

# if RUN_ETA0_T_WIDTH_VALIDATION_SCAN:
#     T_values_validation_scan = [10*ns, 20*ns]
#     width_factor_ranges_validation_scan = [
#         (0.05, 1.50),
#         (0.10, 1.00),
#         (0.20, 1.00),
#         (0.05, 3.00),
#     ]

#     eta0_T_width_validation_scan_df = scan_eta0_T_and_width_ranges(
#         T_values=T_values_validation_scan,
#         width_factor_ranges=width_factor_ranges_validation_scan,
#         n_restarts_per_setting=10,
#         seed_base=2026,
#         success_threshold=gaussian_grape_success_threshold,
#     )
# else:
#     print('Eta=0 T/width validation scan not started. Set RUN_ETA0_T_WIDTH_VALIDATION_SCAN = True to run.')
